# Zero-Day Detection — Leave-One-Out Experiment

Test xem Nhánh 2 (anomaly detection) có phát hiện được dạng SQLi chưa từng thấy không.

In [4]:
import json
from pathlib import Path

# Xác định project root (notebook chạy từ notebooks/)
project_root = Path().resolve().parent
report_path = project_root / "reports" / "zeroday_experiment" / "summary.json"

with open(report_path) as f:
    data = json.load(f)

res = data["results"]
baseline = data["baseline"]
print("Baseline: FPR={:.2%}, DR (all anomalous)={:.2%}".format(baseline["fpr"], baseline["dr_all_anomalous"]))

Baseline: FPR=0.50%, DR (all anomalous)=23.21%


In [5]:
import pandas as pd

rows = []
for name, r in res.items():
    rows.append({
        "Excluded label": name,
        "B1 F1-macro": r["branch1_f1_macro"],
        "B1 miss rate": r["branch1_miss_rate"],
        "B2 DR": r["branch2_detection_rate"],
        "Combined coverage": r["combined_coverage"],
        "N test": r["n_test_samples"],
        "B1 prediction distro": str(r["branch1_prediction_distribution"]),
    })

df = pd.DataFrame(rows).sort_values("B2 DR", ascending=False)
df.style.format({
    "B1 F1-macro": "{:.4f}",
    "B1 miss rate": "{:.2%}",
    "B2 DR": "{:.2%}",
    "Combined coverage": "{:.2%}",
    "N test": "{:,.0f}",
})

,Excluded label,B1 F1-macro,B1 miss rate,B2 DR,Combined coverage,N test,B1 prediction distro
1,error_based,0.9784,0.00%,89.68%,89.68%,"1,560","{'boolean_blind': 1165, 'union_based': 395}"
3,time_blind,0.9773,0.27%,12.73%,12.97%,"3,000","{'boolean_blind': 2992, 'normal': 8}"
2,boolean_blind,0.9973,90.20%,5.40%,94.00%,"3,000","{'normal': 2706, 'time_blind': 277, 'error_based': 10, 'union_based': 7}"
0,union_based,0.9803,2.47%,0.53%,2.97%,"3,000","{'boolean_blind': 2923, 'normal': 74, 'error_based': 3}"


## Phát hiện chính

- **error_based** → B2 bắt **~90%** — zero-day detection hiệu quả nhất
- **boolean_blind** → B1 miss **90%**, B2 chỉ **5.4%** — điểm yếu nhất, cần feature engineering riêng
- **union_based** + **time_blind** → B2 dưới baseline (0.5% và 12.7% vs 23.2%)

### Kết luận
Nhánh 2 **có zero-day capability cho error_based attacks**. Để cải thiện các lớp còn lại cần:
- Threshold tuning (Balanced: FPR=1% cho DR=25%)
- Feature engineering thêm (token-level features, query structure graph)
- Kết hợp cả 2 nhánh (combined coverage cao nhất: boolean_blind 94%)